In [ ]:
#1 battery  ## CHANGE 
#full data set
#start empty end empty 
# Account b  ## CHANGE 
## outputs csv schedule ## CHANGE NAME of csv
import pandas as pd
import gurobipy as gp
from gurobipy import GRB


df2 = pd.read_csv("battery_dataset_20260327_cleaned.csv") 


##CHANGE
df = df2[df2["AccountNumber"] == "account_b"]

df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")
df = df.dropna(subset=["Datetime"]).copy()

df["date_only"] = df["Datetime"].dt.date
df["hour"] = df["Datetime"].dt.hour

# Use full dataset
df_model = df.copy()

# Map all days in the dataset
all_days = sorted(df_model["date_only"].unique())
day_map = {day: i + 1 for i, day in enumerate(all_days)}

df_model["d"] = df_model["date_only"].map(day_map)
df_model["t"] = df_model["hour"]

# Aggregate to hourly level
hourly = (
    df_model.groupby(["d", "t"], as_index=False)
    .agg({
        "Quantity": "sum",
        "Price": "mean"   
    })
)

Price = {
    (int(row["d"]), int(row["t"])): float(row["Price"])
    for _, row in hourly.iterrows()
}
Quantity = {
    (int(row["d"]), int(row["t"])): float(row["Quantity"])
    for _, row in hourly.iterrows()
}

DT = sorted(Quantity.keys())
DT_sorted = sorted(DT)

# Parameters 

##CHANGE
num_batteries = 1
cap = 13.5
r_ch = 5.0
r_dis = 11.5
eta_ch = 0.95
eta_dis = 0.95

total_cap = cap * num_batteries
total_r_ch = r_ch * num_batteries
total_r_dis = r_dis * num_batteries


original_cost = sum(
    Quantity[d, t] * Price[d, t]
    for d, t in DT
)

## MODEL
m = gp.Model("battery_schedule_start_empty_end_empty")

# Decision variables
grid = m.addVars(DT, lb=0, vtype=GRB.CONTINUOUS, name="grid")
charge = m.addVars(DT, lb=0, vtype=GRB.CONTINUOUS, name="charge")
discharge = m.addVars(DT, lb=0, vtype=GRB.CONTINUOUS, name="discharge")
soc = m.addVars(DT, lb=0, ub=total_cap, vtype=GRB.CONTINUOUS, name="soc")
z = m.addVars(DT, vtype=GRB.BINARY, name="mode")


# Constraints

# Energy balance
for d, t in DT:
    m.addConstr(
        grid[d, t] + discharge[d, t] == Quantity[d, t] + charge[d, t],
        name=f"balance_d{d}_t{t}"
    )

# Cannot charge and discharge at the same time
for d, t in DT:
    m.addConstr(
        charge[d, t] <= total_r_ch * z[d, t],
        name=f"charge_limit_d{d}_t{t}"
    )
    m.addConstr(
        discharge[d, t] <= total_r_dis * (1 - z[d, t]),
        name=f"discharge_limit_d{d}_t{t}"
    )

# SOC evolution
for i, (d, t) in enumerate(DT_sorted):
    if i == 0:
        m.addConstr(
            soc[d, t] == 0 + eta_ch * charge[d, t] - discharge[d, t] / eta_dis,
            name=f"soc_init_d{d}_t{t}"
        )
    else:
        prev_d, prev_t = DT_sorted[i - 1]
        m.addConstr(
            soc[d, t] == soc[prev_d, prev_t] + eta_ch * charge[d, t] - discharge[d, t] / eta_dis,
            name=f"soc_flow_d{d}_t{t}"
        )
# end batteries as empty 
last_d, last_t = DT_sorted[-1]
m.addConstr(
    soc[last_d, last_t] == 0,
    name="end_empty"
)


# Objective function 
grid_cost = gp.quicksum(
    grid[d, t] * Price[d, t]
    for d, t in DT
)

m.setObjective(grid_cost, GRB.MINIMIZE)


m.optimize()


# Results
if m.status == GRB.OPTIMAL:
    optimized_grid_cost = sum(
        grid[d, t].X * Price[d, t]
        for d, t in DT
    )
    savings = original_cost - optimized_grid_cost

    print("\nOptimal Solution")
    print(f"Fixed number of batteries = {num_batteries}")
    print("Start SOC = 0.0000 kWh")
    print("End SOC = 0.0000 kWh")
    print(f"Original cost without battery = ${original_cost:,.2f}")
    print(f"Optimized cost with battery schedule = ${optimized_grid_cost:,.2f}")
    print(f"Energy bill savings = ${savings:,.2f}")

    print("\nBattery activity:")
    for d, t in DT_sorted:
        ch = charge[d, t].X
        dis = discharge[d, t].X
        s = soc[d, t].X
        p = Price[d, t]
        g = grid[d, t].X

        if ch > 1e-6 or dis > 1e-6:
            action = "CHARGE" if ch > 1e-6 else "DISCHARGE"
            print(
                f"Day {d:3d}, Hour {t:02d} | "
                f"Price={p:.4f} | Grid={g:.4f} | "
                f"Charge={ch:.4f} | Discharge={dis:.4f} | "
                f"SOC={s:.4f} | Action={action}"
            )
else:
    print("No optimal solution found.") 

if m.status == GRB.OPTIMAL:
    reverse_day_map = {v: k for k, v in day_map.items()}

    schedule_rows = []

    for d, t in DT_sorted:
        charge_val = charge[d, t].X
        discharge_val = discharge[d, t].X

        if charge_val <= 1e-6 and discharge_val <= 1e-6:
            continue

        actual_date = pd.to_datetime(reverse_day_map[d])
        actual_datetime = actual_date + pd.Timedelta(hours=int(t))

        price = Price[d, t]
        demand = Quantity[d, t]
        grid_val = grid[d, t].X
        soc_val = soc[d, t].X

        if charge_val > 1e-6:
            action = "CHARGE"
        else:
            action = "DISCHARGE"

        net_battery_output = discharge_val - charge_val
        grid_cost = grid_val * price
        original_cost_hour = demand * price
        hourly_savings = original_cost_hour - grid_cost

        schedule_rows.append({
            "Datetime": actual_datetime,
            "Price": price,
            "Demand": demand,
            "Grid": grid_val,
            "Charge": charge_val,
            "Discharge": discharge_val,
            "SOC": soc_val,
            "Action": action,
            "Net_Battery_Output": net_battery_output,
            "Grid_Cost": grid_cost,
            "Original_Cost": original_cost_hour,
            "Hourly_Savings": hourly_savings
        })

    schedule_df = pd.DataFrame(schedule_rows)
    schedule_df = schedule_df.sort_values("Datetime").reset_index(drop=True)

    schedule_df = schedule_df.round(4)

    # Save CSV 

    ## CHANGE CSV name to match number of battereis and account 
    schedule_df.to_csv("battery_schedule_AcctB_fulldataset.csv", index=False)

    print("\nschedule saved")